# Notebook 1: OpenAI Agents SDK — Basics

**What you'll learn:** What is the Agents SDK, why it exists, its core primitives, and your first agent in 5 lines.

---

## What is an AI Agent?

An **AI Agent** is NOT just a chatbot. It is:

```
Agent = LLM + Instructions + Tools + Decision-Making Loop
```

A chatbot answers questions. An agent **takes actions**, **makes decisions**, and **completes tasks autonomously**.

### Real-World Analogy
- **Chatbot** = A librarian who answers your questions
- **Agent** = A personal assistant who books flights, sends emails, checks weather, and plans your day

---

## Why OpenAI Agents SDK?

| Feature | Raw LLM API | LangChain | **Agents SDK** |
|---|---|---|---|
| Complexity | Manual everything | Heavy abstractions | **Minimal primitives** |
| Learning Curve | Low (but manual) | Steep | **Gentle** |
| Production Ready | DIY | Yes | **Yes** |
| Tool Calling | Manual JSON | Chain-based | **Decorator-based** |
| Multi-Agent | Build yourself | Complex graphs | **Built-in handoffs** |
| Provider Lock-in | OpenAI only | Any | **Any (100+ LLMs)** |

The SDK evolved from OpenAI's experimental **Swarm** project into a production-grade framework.

---

## The 4 Core Primitives

The entire SDK is built on just **4 concepts**:

```
1. Agent        → The brain (LLM + instructions + tools)
2. Runner       → The executor (runs the agent loop)
3. Tools        → The hands (functions the agent can call)
4. Handoffs     → The delegation (agent-to-agent transfer)
```

That's it. No graphs, no nodes, no edges, no state machines.

## Installation

In [1]:
# Install the SDK
# uv add openai-agents

## Setup API Key

By default, the SDK uses OpenAI. Set your key:

In [2]:
import os

# Option 1: Set directly (for learning only — never commit real keys!)
# os.environ["OPENAI_API_KEY"] = "sk-your-key-here"

os.environ["MODEL_KEY"]="ollama"

# Option 2: Use .env file (recommended)
# uv add python-dotenv
from dotenv import load_dotenv
load_dotenv()

True

---

## Your First Agent — Hello World

In [3]:
from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel,set_tracing_disabled

set_tracing_disabled(True)

local_model = OpenAIChatCompletionsModel(
    model="qwen3.5:cloud ",
    openai_client=AsyncOpenAI(
        api_key="ollama",
        base_url="http://localhost:11434/v1",
    ),
)
agent = Agent(
    name="Greeter",
    instructions="You are a friendly assistant. Be concise.",
    model=local_model
)
    


result = await Runner.run(agent, "Hello! What can you do?")

print(result.final_output)

Hello! I can help with writing, answering questions, summarizing, translating, coding, and brainstorming. What would you like to work on today?


### Breaking it down:

```python
Agent(name, instructions)   # WHO the agent is
Runner.run_sync(agent, input) # RUN the agent with user input  
result.final_output           # GET what the agent produced
```

**That's the entire mental model.** Everything else builds on this.

---

## Async Version (For Jupyter Notebooks)

`Runner.run_sync()` doesn't work inside Jupyter because Jupyter already has an event loop running.  
Use `await Runner.run()` instead:

In [4]:
from agents import Agent, Runner

agent = Agent(
    name="Greeter",
    instructions="You are a friendly assistant. Be concise.",
    model=local_model
)

# In Jupyter, use await directly (no asyncio.run needed)
result = await Runner.run(agent, "What is the capital of Pakistan?")
print(result.final_output)

The capital of Pakistan is Islamabad.


---

## Understanding the Result Object

The `RunResult` object contains much more than just the output:

In [5]:
from agents import Agent, Runner

agent = Agent(name="Analyst", instructions="You analyze topics briefly.",model=local_model)

result = await Runner.run(agent, "Why is Python popular for AI?")

# The key properties of RunResult:
print("=" * 50)
print(f"Final Output:  {result.final_output[:100]}...")   # The text response
print(f"Last Agent:    {result.last_agent.name}")          # Which agent finished
print(f"New Items:     {len(result.new_items)}")           # Items generated during run
print(f"Raw Responses: {len(result.raw_responses)}")       # Raw LLM responses
print("=" * 50)

Final Output:  Python is the leading language for AI due to four key factors:

*   **Rich Ecosystem:** Extensive li...
Last Agent:    Analyst
New Items:     1
Raw Responses: 1


---

## The Agent Loop — How It Actually Works

When you call `Runner.run()`, here's what happens internally:

```
┌─────────────────────────────────────────────┐
│           Runner.run(agent, input)          │
│                                             │
│  ┌──────────────────────────────────────┐   │
│  │ 1. Send input + instructions to LLM  │   │
│  │ 2. LLM responds                      │   │
│  │ 3. Check response:                   │   │
│  │    ├─ Final output? → RETURN result  │   │
│  │    ├─ Tool call?    → Execute tool,  │   │
│  │    │                  loop back to 1 │   │
│  │    └─ Handoff?      → Switch agent,  │   │
│  │                       loop back to 1 │   │
│  └──────────────────────────────────────┘   │
│                                             │
│  This loop IS the "agentic" behavior.       │
│  The LLM DECIDES what to do next.           │
└─────────────────────────────────────────────┘
```

**Key insight:** The agent doesn't just answer — it enters a **reasoning loop** where it can call tools, get results, reason again, and repeat until done.

---

## Multi-Turn Conversations

Agents can maintain conversation context across turns using `to_input_list()`:

In [7]:
from agents import Agent, Runner

agent = Agent(
    name="Tutor",
    instructions="You are a Python tutor. Be concise. Remember context from previous messages.",
    model=local_model
)

# Turn 1
result = await Runner.run(agent, "What is a list in Python?")
print(f"Turn 1: {result.final_output}\n")

# Turn 2 — carry conversation history forward
new_input = result.to_input_list() + [{"role": "user", "content": "How do I sort one?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 2: {result.final_output}\n")

# Turn 3
new_input = result.to_input_list() + [{"role": "user", "content": "What about reverse sort?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 3: {result.final_output}")

Turn 1: A **list** is a built-in Python data structure used to store an ordered collection of items.

Key features:
1.  **Ordered:** Items maintain their position.
2.  **Mutable:** You can change, add, or remove items after creation.
3.  **Flexible:** Can contain mixed data types (integers, strings, etc.).

Example:
```python
numbers = [1, 2, 3]
numbers[0] = 10  # Changes the first item to 10
numbers.append(4)  # Adds 4 to the end
```

Turn 2: There are two main ways to sort a list:

1.  **`.sort()`**: Sorts the list **in-place** (modifies the original).
2.  **`sorted()`**: Returns a **new** sorted list (keeps the original unchanged).

Example:
```python
nums = [3, 1, 2]

nums.sort()              # nums becomes [1, 2, 3]
new_nums = sorted(nums)  # Creates a new sorted list
```

To sort in descending order, add `reverse=True`:
```python
nums.sort(reverse=True)  # [3, 2, 1]
```

Turn 3: There are two distinct operations:

1.  **Sort Descending** (High to Low):
    Use the `reverse=True` 

### Key Pattern:
```python
# This is how you maintain conversation memory:
new_input = previous_result.to_input_list() + [{"role": "user", "content": "next message"}]
result = await Runner.run(agent, new_input)
```

---

## 3 Ways to Run an Agent

| Method | Use Case | Async? |
|---|---|---|
| `Runner.run()` | Jupyter, async apps, FastAPI | Yes |
| `Runner.run_sync()` | Scripts, CLI tools | No |
| `Runner.run_streamed()` | Real-time UIs, streaming | Yes |

In [9]:
from agents import Agent, Runner

agent = Agent(name="Streamer", instructions="Explain briefly.",model=local_model)

# Streaming — see tokens as they arrive
result = Runner.run_streamed(agent, "What is machine learning in 2 sentences?")

async for event in result.stream_events():
    if event.type == "raw_response_event" and hasattr(event.data, 'delta'):
        print(event.data.delta, end="", flush=True)

print(f"\n\n--- Final: {result.final_output}")

Thinking Process:

1.  **Analyze the Request:**
    *   Topic: Machine Learning (ML).
    *   Constraint: Explain briefly in exactly 2 sentences.

2.  **Identify Key Concepts of ML:**
    *   Learning from data.
    *   Improving performance over time/experience.
    *   Making predictions or decisions.
    *   Without being explicitly programmed for every specific rule.

3.  **Drafting Sentence 1 (Definition/Core Mechanism):**
    *   *Draft 1:* Machine learning is a field of artificial intelligence where computers learn from data instead of being explicitly programmed.
    *   *Draft 2:* It is a subset of AI that enables systems to automatically learn and improve from experience without being explicitly programmed.
    *   *Selection:* Draft 2 is strong and standard.

4.  **Drafting Sentence 2 (Function/Outcome):**
    *   *Draft 1:* It uses algorithms to find patterns and make predictions or decisions based on new information.
    *   *Draft 2:* By identifying patterns in large data

---

## Exercise: Build Your First Agent

Create an agent that acts as a **Lahore travel guide**. Ask it 3 questions in a multi-turn conversation.

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. Create Agent with name="Lahore Guide" and good instructions
# 2. Run it with a question about Lahore
# 3. Use to_input_list() to continue the conversation

agent=Agent(
    name="Lahore",
    instructions="you are ai assistant with lahore information",
    model=local_model
)
result = await Runner.run(agent, "Tell me about Lahore?")
print(result.final_output)

new_input = result.to_input_list() + [{"role": "user", "content": "How can i explore?"}]
result = await Runner.run(agent, new_input)
print(f"Turn 2: {result.final_output}\n")




Welcome to **Lahore**, the heart of Pakistan! 🇵🇰

As an AI assistant specialized in Lahore information, I can tell you that this city is much more than just a metropolis; it is a living museum of history, a hub of culture, and arguably the food capital of the country. Here is a comprehensive overview of what makes Lahore special:

### 1. **Introduction & Nicknames**
*   **Status:** Capital of the Punjab province and the 2nd largest city in Pakistan.
*   **Nicknames:** Often called the **"City of Gardens"** (due to its historical gardens) and the **"Heart of Pakistan"** (because of its cultural significance).
*   **Vibe:** Lahore is known for its hustle, hospitality, and the famous phrase *"Lahore Lahore Aye"* (Lahore is truly Lahore), meaning the city is unique and unmatched.

### 2. **Historical Significance**
Lahore has over 2,000 years of recorded history. It reached its golden age under the **Mughal Empire** (16th–18th centuries), who left behind architectural marvels. Later, it se

---

## Summary — What You Learned

| Concept | Code |
|---|---|
| Create an agent | `Agent(name, instructions)` |
| Run async | `await Runner.run(agent, input)` |
| Run sync | `Runner.run_sync(agent, input)` |
| Run streaming | `Runner.run_streamed(agent, input)` |
| Get output | `result.final_output` |
| Continue conversation | `result.to_input_list() + [new_msg]` |

**Next:** Notebook 2 — Creating Proper Agents with instructions, output types, and model settings.